# Demo / Ejemplo de uso de la operacion OP-02: Validate trips

Este notebook muestra ejemplos de uso de `validate_trips(...)` en dos escenarios:
1) un caso canónico / limpio, pensado para mostrar el uso normal;
2) un caso más realista con ADATRAP importado, pensado para mostrar uso diagnóstico.

La idea no es testear la implementación, sino mostrar cómo se usa la operación y cómo se interpretan `report.ok`, `issues`, `summary`, `metadata["events"]` y `metadata["is_validated"]`.

### Configuración inicial

In [12]:
from pathlib import Path
import sys

REPO_ROOT = Path("../../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "synthetic"
ADATRAP_DATA_PATH = REPO_ROOT / "data" / "ADATRAP"

In [15]:
import pandas as pd
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

### Import generales

In [2]:
import pandas as pd
import yaml

from pylondrina.importing import ImportOptions, import_trips_from_dataframe
from pylondrina.validation import ValidationOptions, validate_trips
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec

from pylondrina.sources.helpers import import_trips_from_profile
from scripts.source_profiles.factories_adatrap.make_adatrap_trips_profile import (
    make_adatrap_trips_profile,
)
from scripts.source_profiles.factories_adatrap.trips_defaults import (
    ADATRAP_TRIPS_DEFAULT_PROVENANCE_EXAMPLE,
)

### Helpers pequeños para visualizacion

In [3]:
def issues_to_frame(issues):
    rows = []
    for it in issues:
        rows.append({
            "level": it.level,
            "code": it.code,
            "message": it.message,
            "field": it.field,
            "row_count": it.row_count,
        })
    return pd.DataFrame(rows)


def show_validation_result(trips, report, *, title: str):
    print(f"\n=== {title} ===")
    print(f"report.ok: {report.ok}")
    print(f"n_issues: {len(report.issues)}")
    print(f"is_validated: {trips.metadata.get('is_validated')}")
    print("\nsummary:")
    display(pd.Series(report.summary))

    print("\núltimo evento:")
    display(trips.metadata["events"][-1])

    if report.issues:
        print("\nissues:")
        display(issues_to_frame(report.issues))
    else:
        print("\nissues: []")

## Caso 1. Ejemplo canónico / limpio

Este caso usa el CSV sintético limpio que se genero previamente.
La idea es mostrar el uso más directo: cargar fuente, importar, validar e inspeccionar el resultado.

### Schema canonico para la demo limpia

In [4]:
CANONICAL_MODE_VALUES = [
    "walk",
    "bicycle",
    "scooter",
    "motorcycle",
    "car",
    "taxi",
    "ride_hailing",
    "bus",
    "metro",
    "train",
    "other",
]

CANONICAL_PURPOSE_VALUES = [
    "home",
    "work",
    "education",
    "shopping",
    "errand",
    "health",
    "leisure",
    "transfer",
    "other",
]

CANONICAL_GENDER_VALUES = ["female", "male", "other", "unknown"]

schema_demo_clean = TripSchema(
    version="1.1-demo-clean",
    fields={
        "movement_id": FieldSpec("movement_id", "string", required=True),
        "user_id": FieldSpec("user_id", "string", required=True),

        "origin_longitude": FieldSpec(
            "origin_longitude",
            "float",
            required=True,
            constraints={"range": {"min": -180.0, "max": 180.0}},
        ),
        "origin_latitude": FieldSpec(
            "origin_latitude",
            "float",
            required=True,
            constraints={"range": {"min": -90.0, "max": 90.0}},
        ),
        "destination_longitude": FieldSpec(
            "destination_longitude",
            "float",
            required=True,
            constraints={"range": {"min": -180.0, "max": 180.0}},
        ),
        "destination_latitude": FieldSpec(
            "destination_latitude",
            "float",
            required=True,
            constraints={"range": {"min": -90.0, "max": 90.0}},
        ),

        "origin_h3_index": FieldSpec(
            "origin_h3_index",
            "string",
            required=True,
            constraints={"h3": {"require_valid": True}},
        ),
        "destination_h3_index": FieldSpec(
            "destination_h3_index",
            "string",
            required=True,
            constraints={"h3": {"require_valid": True}},
        ),

        "trip_id": FieldSpec("trip_id", "string", required=True),
        "movement_seq": FieldSpec(
            "movement_seq",
            "int",
            required=True,
            constraints={"range": {"min": 0}},
        ),

        "origin_time_utc": FieldSpec("origin_time_utc", "datetime", required=False),
        "destination_time_utc": FieldSpec("destination_time_utc", "datetime", required=False),

        "mode": FieldSpec(
            "mode",
            "categorical",
            required=False,
            domain=DomainSpec(values=CANONICAL_MODE_VALUES, extendable=False),
        ),
        "purpose": FieldSpec(
            "purpose",
            "categorical",
            required=False,
            domain=DomainSpec(values=CANONICAL_PURPOSE_VALUES, extendable=False),
        ),
        "user_gender": FieldSpec(
            "user_gender",
            "categorical",
            required=False,
            domain=DomainSpec(values=CANONICAL_GENDER_VALUES, extendable=False),
        ),

        "origin_municipality": FieldSpec("origin_municipality", "string", required=False),
        "destination_municipality": FieldSpec("destination_municipality", "string", required=False),
        "trip_weight": FieldSpec("trip_weight", "float", required=False),
        "timezone_offset_min": FieldSpec("timezone_offset_min", "int", required=False),
    },
    required=[
        "movement_id",
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "origin_h3_index",
        "destination_h3_index",
        "trip_id",
        "movement_seq",
    ],
    semantic_rules=None,
)

### Se carga dataset fuente limpio

In [7]:
csv_demo_clean = DATA_PATH / "demo_op02_clean.csv"
df_demo_clean = pd.read_csv(csv_demo_clean)

print(df_demo_clean.shape)
display(df_demo_clean.head())

(300, 21)


,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,...,destination_h3_index,mode,purpose,user_gender,origin_municipality,destination_municipality,trip_weight,timezone_offset_min,origin_stop_id,household_vehicle_count
0,m00000,u0092,t00000,0,-70.726988,-33.434123,-70.719845,-33.443185,2026-03-05T22:31:00Z,2026-03-05T23:27:00Z,...,88b2c55519fffff,ride_hailing,health,other,Estación Central,Las Condes,1.096,-180,STOP_4225,0
1,m00001,u0050,t00001,0,-70.766478,-33.442381,-70.806850,-33.423552,2026-03-03T02:57:00Z,2026-03-03T04:31:00Z,...,88b2c54249fffff,metro,education,female,Las Condes,Maipú,3.028,-180,STOP_3746,1
2,m00002,u0038,t00002,0,-70.762472,-33.569227,-70.733115,-33.525176,2026-03-06T00:55:00Z,2026-03-06T02:40:00Z,...,88b2c5470dfffff,metro,shopping,other,Pudahuel,Las Condes,1.169,-180,STOP_3005,1
3,m00003,u0078,t00003,0,-70.688232,-33.507355,-70.644262,-33.517737,2026-03-02T14:54:00Z,2026-03-02T15:54:00Z,...,88b2c54657fffff,other,transfer,other,Santiago,Pudahuel,3.929,-180,STOP_4896,0
4,m00004,u0025,t00004,0,-70.782516,-33.484129,-70.749178,-33.451496,2026-03-01T13:23:00Z,2026-03-01T14:48:00Z,...,88b2c5553dfffff,scooter,errand,unknown,Ñuñoa,Quilicura,0.517,-180,STOP_4063,1


### Se importa a TripDataset

In [8]:
import_options_demo_clean = ImportOptions(
    keep_extra_fields=True,
    strict=False,
    strict_domains=False,
    single_stage=False,
)

trips_demo_clean, import_report_demo_clean = import_trips_from_dataframe(
    df_demo_clean,
    schema=schema_demo_clean,
    source_name="demo_validate_trips_clean",
    options=import_options_demo_clean,
    provenance={
        "source": {
            "name": "synthetic_demo_clean",
            "entity": "trips",
            "version": "usage_example_v1",
        }
    },
    h3_resolution=8,
)

print("Import ok:", import_report_demo_clean.ok)
print("Eventos tras import:", [e["op"] for e in trips_demo_clean.metadata.get("events", [])])
display(trips_demo_clean.data.head())
display(import_report_demo_clean.issues)

Import ok: True
Eventos tras import: ['import_trips']


,movement_id,user_id,trip_id,movement_seq,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,...,destination_h3_index,mode,purpose,user_gender,origin_municipality,destination_municipality,trip_weight,timezone_offset_min,origin_stop_id,household_vehicle_count
0,m00000,u0092,t00000,0,-70.726988,-33.434123,-70.719845,-33.443185,2026-03-05 22:31:00+00:00,2026-03-05 23:27:00+00:00,...,88b2c55519fffff,ride_hailing,health,other,Estación Central,Las Condes,1.096,-180,STOP_4225,0
1,m00001,u0050,t00001,0,-70.766478,-33.442381,-70.806850,-33.423552,2026-03-03 02:57:00+00:00,2026-03-03 04:31:00+00:00,...,88b2c54249fffff,metro,education,female,Las Condes,Maipú,3.028,-180,STOP_3746,1
2,m00002,u0038,t00002,0,-70.762472,-33.569227,-70.733115,-33.525176,2026-03-06 00:55:00+00:00,2026-03-06 02:40:00+00:00,...,88b2c5470dfffff,metro,shopping,other,Pudahuel,Las Condes,1.169,-180,STOP_3005,1
3,m00003,u0078,t00003,0,-70.688232,-33.507355,-70.644262,-33.517737,2026-03-02 14:54:00+00:00,2026-03-02 15:54:00+00:00,...,88b2c54657fffff,other,transfer,other,Santiago,Pudahuel,3.929,-180,STOP_4896,0
4,m00004,u0025,t00004,0,-70.782516,-33.484129,-70.749178,-33.451496,2026-03-01 13:23:00+00:00,2026-03-01 14:48:00+00:00,...,88b2c5553dfffff,scooter,errand,unknown,Ñuñoa,Quilicura,0.517,-180,STOP_4063,1


[Issue(level='info', code='IMP.METADATA.DATASET_ID_CREATED', message="Se generó dataset_id para el dataset importado: 'tripds_9bf2d23b5aff4f88b99ac35dffc2c141'.", field=None, source_field=None, row_count=None, details={'dataset_id': 'tripds_9bf2d23b5aff4f88b99ac35dffc2c141', 'generator': 'uuid4_hex', 'stored_in': 'metadata.dataset_id'})]

### Se valida el dataset limpio

In [9]:
validation_options_demo_clean = ValidationOptions(
    strict=False,
    validate_required_fields=True,
    validate_types_and_formats=True,
    validate_constraints=True,
    validate_domains="full",
    validate_temporal_consistency=True,
    validate_duplicates=True,
    duplicates_subset=("user_id", "origin_time_utc", "origin_h3_index", "destination_h3_index"),
    allow_partial_od_spatial=True,
)

report_demo_clean = validate_trips(
    trips_demo_clean,
    options=validation_options_demo_clean,
)

### Se inspecciona el resultado limpio

In [10]:
show_validation_result(
    trips_demo_clean,
    report_demo_clean,
    title="Caso 1 - ejemplo canónico / limpio",
)


=== Caso 1 - ejemplo canónico / limpio ===
report.ok: True
n_issues: 0
is_validated: True

summary:


ok                                                              True
n_rows                                                           300
n_issues                                                           0
n_errors                                                           0
n_warnings                                                         0
n_info                                                             0
counts_by_level                {'error': 0, 'warning': 0, 'info': 0}
counts_by_code                                                    {}
checked_fields     [destination_h3_index, destination_latitude, d...
checks_executed    {'required_fields': True, 'types_and_formats':...
schema_version                                        1.1-demo-clean
domains            {'mode': 'full', 'min_required_ratio': 1.0, 'f...
duplicates         {'evaluated': True, 'duplicates_subset': ['use...
temporal           {'evaluated': True, 'tier': 'tier_1', 'n_check...
limits             {'max_issues': 


último evento:


{'op': 'validate_trips',
 'ts_utc': '2026-04-02T13:54:06Z',
 'parameters': {'strict': False,
  'max_issues': 500,
  'sample_rows_per_issue': 5,
  'validate_required_fields': True,
  'validate_types_and_formats': True,
  'validate_constraints': True,
  'validate_domains': 'full',
  'domains_sample_frac': 0.01,
  'domains_min_in_domain_ratio': 1.0,
  'validate_temporal_consistency': True,
  'validate_duplicates': True,
  'duplicates_subset': ['user_id',
   'origin_time_utc',
   'origin_h3_index',
   'destination_h3_index'],
  'allow_partial_od_spatial': True},
 'summary': {'ok': True,
  'n_rows': 300,
  'n_issues': 0,
  'n_errors': 0,
  'n_warnings': 0,
  'n_info': 0,
  'counts_by_level': {'error': 0, 'warning': 0, 'info': 0},
  'counts_by_code': {},
  'checked_fields': ['destination_h3_index',
   'destination_latitude',
   'destination_longitude',
   'destination_municipality',
   'destination_time_utc',
   'mode',
   'movement_id',
   'movement_seq',
   'origin_h3_index',
   'origin_la


issues: []


### Interpretación breve:

- Este caso muestra el uso normal de validate_trips sobre un TripDataset ya importado.
- Idealmente report.ok debería ser True o, al menos, no contener errores.
- El evento validate_trips queda append-eado en metadata['events'].
- metadata['is_validated'] queda alineado con report.ok.

## Caso 2. Ejemplo con dataset real importado (ADATRAP)

Aquí el objetivo ya no es “mostrar un dataset perfecto”, sino un uso realista de validate_trips como herramienta diagnóstica.
En esta etapa del proyecto, que un dataset real termine con ok=False puede ser completamente esperable y útil.

### Cargar fuente ADATRAP y paraderos

In [13]:
adatrap_viajes = pd.read_csv(
    ADATRAP_DATA_PATH / "semana_2019_05_13.viajes",
    sep="|",
    dtype=str,
    low_memory=False,
    na_values="-",
)

adatrap_stops = pd.read_csv(
    ADATRAP_DATA_PATH / "DIC_777_fixed.csv",
    sep=",",
    dtype=str,
    low_memory=False,
)

print("adatrap_viajes:", adatrap_viajes.shape)
print("adatrap_stops:", adatrap_stops.shape)
display(adatrap_viajes.head())

adatrap_viajes: (1000000, 109)
adatrap_stops: (11211, 8)


,id,nviaje,netapa,etapas,netapassinbajada,ultimaetapaconbajada,tviaje_seg,tviaje_min,dviajeeuclidiana_mts,dviajeenruta_mts,...,tipotransporte_4ta,tesperaest_1era,tesperaest_2da,tesperaest_3era,tesperaest_4ta,escolar,tviaje_en_vehiculo_min,tipo_corte_etapa_viaje,proposito,dviaje_buses
0,4306298,1,2,NaN,0,1,3098.0000,51.6333,15836.0068,25695.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,48.8667,SM2H,TRABAJO,NaN
1,8398026,2,1,NaN,0,1,701.0000,11.6833,3872.8728,4084.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,11.6833,SM2H,TRABAJO,NaN
2,9173978,1,1,NaN,0,1,690.0000,11.5000,4511.1367,4513.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,11.5000,MS_M_M,OTROS,NaN
3,9284970,1,1,NaN,0,1,1209.0000,20.1500,3875.6917,5234.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,20.1500,SM2H,TRABAJO,NaN
4,9403002,1,1,NaN,0,1,1320.0000,22.0000,9128.2852,9258.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,22.0000,M3B,OTROS,NaN


### Construir perfil trips de ADATRAP

Esta parte sigue la ruta de trips/viajes del notebook de ADATRAP: perfil de trips, stage_layout.yaml, domains.yaml y stops para resolver coordenadas.

In [16]:
profile_adatrap_trips = make_adatrap_trips_profile(
    df_stops=adatrap_stops,
    stage_layout_yaml=ADATRAP_DATA_PATH / "stage_layout.yaml",
    domains_yaml=ADATRAP_DATA_PATH / "domains.yaml",
)

df_test = profile_adatrap_trips.preprocess(adatrap_viajes.head(1000))
display(df_test.head())

,id,nviaje,netapa,etapas,netapassinbajada,ultimaetapaconbajada,tviaje_seg,tviaje_min,dviajeeuclidiana_mts,dviajeenruta_mts,paraderosubida,paraderobajada,comunasubida,comunabajada,diseno777subida,diseno777bajada,tiemposubida,tiempobajada,periodosubida,periodobajada,tipodia,mediahora,contrato,factorexpansion,tiempomediodeviaje,periodomediodeviaje,mediahoramediodeviaje,tipodiamediodeviaje,escolar,tviaje_en_vehiculo_min,tipo_corte_etapa_viaje,proposito,dviaje_buses,etapas_detectadas,linea_metro_subida,zona777subida,linea_metro_bajada,zona777bajada,subida_lon,subida_lat,bajada_lon,bajada_lat
0,4306298,1,2,NaN,0,1,3098.0000,51.6333,15836.0068,25695.0000,T-13-104-PO-15,NUBLE,MAIPU,NUNOA,172,299,2019-05-13 06:00:55,2019-05-13 06:52:33,03 - TRANSICION NOCTURNO,04 - PUNTA MANANA,LABORAL,06:00:00,NaN,1.4376,2019-05-13 06:26:44,03 - TRANSICION NOCTURNO,06:00:00,LABORAL,NaN,48.8667,SM2H,TRABAJO,NaN,2,L5,172,L5,299,-70.783226,-33.524734,-70.626419,-33.467472
1,8398026,2,1,NaN,0,1,701.0000,11.6833,3872.8728,4084.0000,T-18-117-NS-5,T-18-153-PO-35,NUNOA,NUNOA,231,240,2019-05-13 06:53:45,2019-05-13 07:05:26,04 - PUNTA MANANA,04 - PUNTA MANANA,LABORAL,06:30:00,NaN,1.2547,2019-05-13 06:59:35,04 - PUNTA MANANA,06:30:00,LABORAL,NaN,11.6833,SM2H,TRABAJO,NaN,1,<NA>,231,<NA>,240,-70.629251,-33.452086,-70.587011,-33.457846
2,9173978,1,1,NaN,0,1,690.0000,11.5000,4511.1367,4513.0000,SANTA LUCIA,SAN ALBERTO HURTADO,SANTIAGO,ESTACION CENTRAL,286,77,2019-05-13 18:12:35,2019-05-13 18:24:05,09 - PUNTA TARDE,09 - PUNTA TARDE,LABORAL,18:00:00,NaN,1.2976,2019-05-13 18:18:20,09 - PUNTA TARDE,18:00:00,LABORAL,NaN,11.5000,MS_M_M,OTROS,NaN,1,L1,286,L1,77,-70.645817,-33.442849,-70.692442,-33.454116
3,9284970,1,1,NaN,0,1,1209.0000,20.1500,3875.6917,5234.0000,T-17-149-SN-8,T-17-140-OP-35,LAS CONDES,LAS CONDES,775,207,2019-05-13 06:56:49,2019-05-13 07:16:58,04 - PUNTA MANANA,04 - PUNTA MANANA,LABORAL,06:30:00,NaN,1.4500,2019-05-13 07:06:53,04 - PUNTA MANANA,07:00:00,LABORAL,NaN,20.1500,SM2H,TRABAJO,NaN,1,<NA>,775,<NA>,207,-70.529608,-33.420266,-70.569059,-33.409114
4,9403002,1,1,NaN,0,1,1320.0000,22.0000,9128.2852,9258.0000,ESCUELA MILITAR,UNION LATINO AMERICANA,LAS CONDES,SANTIAGO,215,280,2019-05-13 16:08:12,2019-05-13 16:30:12,08 - FUERA DE PUNTA TARDE,08 - FUERA DE PUNTA TARDE,LABORAL,16:00:00,NaN,1.2666,2019-05-13 16:19:12,08 - FUERA DE PUNTA TARDE,16:00:00,LABORAL,NaN,22.0000,M3B,OTROS,NaN,1,L1,215,L1,280,-70.584445,-33.414320,-70.673213,-33.449471


### Importar ADATRAP como trips

In [17]:
trips_adatrap, report_adatrap_import = import_trips_from_profile(
    profile=profile_adatrap_trips,
    df=adatrap_viajes.head(5000),
    source_name="ADATRAP trips - usage example",
    provenance=ADATRAP_TRIPS_DEFAULT_PROVENANCE_EXAMPLE,
    h3_resolution=8,
)

print("Import ok:", report_adatrap_import.ok)
print("Eventos tras import:", [e["op"] for e in trips_adatrap.metadata.get("events", [])])

display(trips_adatrap.data.head())
display(issues_to_frame(report_adatrap_import.issues))

Import ok: True
Eventos tras import: ['import_trips']


,movement_id,trip_id,movement_seq,user_id,nviaje,netapa,etapas,netapassinbajada,ultimaetapaconbajada,tviaje_seg,tviaje_min,dviajeeuclidiana_mts,dviajeenruta_mts,origin_stop_code,destination_stop_code,origin_municipality,destination_municipality,diseno777subida,diseno777bajada,origin_time_utc,destination_time_utc,periodosubida,periodobajada,day_type,mediahora,contrato,trip_weight,tiempomediodeviaje,time_period,mediahoramediodeviaje,tipodiamediodeviaje,escolar,tviaje_en_vehiculo_min,tipo_corte_etapa_viaje,purpose,dviaje_buses,etapas_detectadas,linea_metro_subida,zona777subida,linea_metro_bajada,zona777bajada,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index
0,m0,m0,0,4306298,1,2,NaN,0,1,3098.0000,51.6333,15836.0068,25695.0000,T-13-104-PO-15,NUBLE,MAIPU,NUNOA,172,299,2019-05-13 10:00:55+00:00,2019-05-13 10:52:33+00:00,03 - TRANSICION NOCTURNO,04 - PUNTA MANANA,weekday,06:00:00,NaN,1.4376,2019-05-13 06:26:44,night,06:00:00,LABORAL,NaN,48.8667,SM2H,work,NaN,2,L5,172,L5,299,-70.783226,-33.524734,-70.626419,-33.467472,88b2c540ddfffff,88b2c5548bfffff
1,m1,m1,0,8398026,2,1,NaN,0,1,701.0000,11.6833,3872.8728,4084.0000,T-18-117-NS-5,T-18-153-PO-35,NUNOA,NUNOA,231,240,2019-05-13 10:53:45+00:00,2019-05-13 11:05:26+00:00,04 - PUNTA MANANA,04 - PUNTA MANANA,weekday,06:30:00,NaN,1.2547,2019-05-13 06:59:35,morning,06:30:00,LABORAL,NaN,11.6833,SM2H,work,NaN,1,<NA>,231,<NA>,240,-70.629251,-33.452086,-70.587011,-33.457846,88b2c554c3fffff,88b2c50b29fffff
2,m2,m2,0,9173978,1,1,NaN,0,1,690.0000,11.5000,4511.1367,4513.0000,SANTA LUCIA,SAN ALBERTO HURTADO,SANTIAGO,ESTACION CENTRAL,286,77,2019-05-13 22:12:35+00:00,2019-05-13 22:24:05+00:00,09 - PUNTA TARDE,09 - PUNTA TARDE,weekday,18:00:00,NaN,1.2976,2019-05-13 18:18:20,afternoon,18:00:00,LABORAL,NaN,11.5000,MS_M_M,other,NaN,1,L1,286,L1,77,-70.645817,-33.442849,-70.692442,-33.454116,88b2c55413fffff,88b2c555cbfffff
3,m3,m3,0,9284970,1,1,NaN,0,1,1209.0000,20.1500,3875.6917,5234.0000,T-17-149-SN-8,T-17-140-OP-35,LAS CONDES,LAS CONDES,775,207,2019-05-13 10:56:49+00:00,2019-05-13 11:16:58+00:00,04 - PUNTA MANANA,04 - PUNTA MANANA,weekday,06:30:00,NaN,1.4500,2019-05-13 07:06:53,morning,07:00:00,LABORAL,NaN,20.1500,SM2H,work,NaN,1,<NA>,775,<NA>,207,-70.529608,-33.420266,-70.569059,-33.409114,88b2c51995fffff,88b2c519adfffff
4,m4,m4,0,9403002,1,1,NaN,0,1,1320.0000,22.0000,9128.2852,9258.0000,ESCUELA MILITAR,UNION LATINO AMERICANA,LAS CONDES,SANTIAGO,215,280,2019-05-13 20:08:12+00:00,2019-05-13 20:30:12+00:00,08 - FUERA DE PUNTA TARDE,08 - FUERA DE PUNTA TARDE,weekday,16:00:00,NaN,1.2666,2019-05-13 16:19:12,afternoon,16:00:00,LABORAL,NaN,22.0000,M3B,other,NaN,1,L1,215,L1,280,-70.584445,-33.414320,-70.673213,-33.449471,88b2c556d3fffff,88b2c5543bfffff


,level,code,message,field,row_count
0,info,IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND,El campo opcional 'mode' no está presente en la fuente y no se encontró correspondencia; se omitirá.,mode,NaN
1,info,IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND,El campo opcional 'user_gender' no está presente en la fuente y no se encontró correspondencia; se omitirá.,user_gender,NaN
2,info,IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND,El campo opcional 'user_age_group' no está presente en la fuente y no se encontró correspondencia; se omitirá.,user_age_group,NaN
3,info,IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND,El campo opcional 'income_quintile' no está presente en la fuente y no se encontró correspondencia; se omitirá.,income_quintile,NaN
4,info,IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND,El campo opcional 'origin_time_local_hhmm' no está presente en la fuente y no se encontró correspondencia; se omitirá.,origin_time_local_hhmm,NaN
5,info,IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND,El campo opcional 'destination_time_local_hhmm' no está presente en la fuente y no se encontró correspondencia; se o...,destination_time_local_hhmm,NaN
6,info,IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND,El campo opcional 'mode_sequence' no está presente en la fuente y no se encontró correspondencia; se omitirá.,mode_sequence,NaN
7,info,IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND,El campo opcional 'periodomediodeviaje' no está presente en la fuente y no se encontró correspondencia; se omitirá.,periodomediodeviaje,NaN
8,warning,IMP.TYPE.COERCE_PARTIAL,La conversión mínima del campo 'destination_time_utc' falló en algunas filas (13.2%); se marcarán como nulos para va...,destination_time_utc,662.0
9,info,IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND,El campo opcional 'mode' no está presente en la fuente y no se encontró correspondencia; se omitirá.,mode,NaN


### Verificación mínima antes de validar

In [18]:
print("schema_effective existe:", trips_adatrap.schema_effective is not None)
print("metadata['domains_effective'] existe:", "domains_effective" in trips_adatrap.metadata)
print("metadata['temporal'] existe:", "temporal" in trips_adatrap.metadata)
print("metadata['h3'] existe:", "h3" in trips_adatrap.metadata)

display(pd.Series({
    "temporal": trips_adatrap.metadata.get("temporal"),
    "h3": trips_adatrap.metadata.get("h3"),
}))

schema_effective existe: True
metadata['domains_effective'] existe: True
metadata['temporal'] existe: True
metadata['h3'] existe: True


temporal    {'tier': 'tier_1', 'fields_present': ['origin_time_utc', 'destination_time_utc'], 'normalization': {'origin_time_utc...
h3          {'resolution': 8, 'source_fields': [['origin_latitude', 'origin_longitude'], ['destination_latitude', 'destination_l...
dtype: object

### Validar ADATRAP importado

In [19]:
validation_options_adatrap = ValidationOptions(
    strict=False,
    validate_required_fields=True,
    validate_types_and_formats=True,
    validate_constraints=True,
    validate_domains="full",
    validate_temporal_consistency=True,
    validate_duplicates=False,
    allow_partial_od_spatial=True,
)

report_adatrap_validate = validate_trips(
    trips_adatrap,
    options=validation_options_adatrap,
)

### Inspeccionar el resultado de ADATRAP

In [20]:
show_validation_result(
    trips_adatrap,
    report_adatrap_validate,
    title="Caso 2 - ADATRAP importado (uso diagnóstico)",
)


=== Caso 2 - ADATRAP importado (uso diagnóstico) ===
report.ok: False
n_issues: 4
is_validated: False

summary:


ok                                                                                                                                   False
n_rows                                                                                                                                5000
n_issues                                                                                                                                 4
n_errors                                                                                                                                 4
n_warnings                                                                                                                               0
n_info                                                                                                                                   0
counts_by_level                                                                                      {'error': 4, 'warning': 0, 'info': 0}
counts_by_code     {'VAL.CO


último evento:


{'op': 'validate_trips',
 'ts_utc': '2026-04-02T14:11:55Z',
 'parameters': {'strict': False,
  'max_issues': 500,
  'sample_rows_per_issue': 5,
  'validate_required_fields': True,
  'validate_types_and_formats': True,
  'validate_constraints': True,
  'validate_domains': 'full',
  'domains_sample_frac': 0.01,
  'domains_min_in_domain_ratio': 1.0,
  'validate_temporal_consistency': True,
  'validate_duplicates': False,
  'duplicates_subset': None,
  'allow_partial_od_spatial': True},
 'summary': {'ok': False,
  'n_rows': 5000,
  'n_issues': 4,
  'n_errors': 4,
  'n_warnings': 0,
  'n_info': 0,
  'counts_by_level': {'error': 4, 'warning': 0, 'info': 0},
  'counts_by_code': {'VAL.CORE.OD_SPATIAL_BOTH_MISSING': 1,
   'VAL.CORE.NULLABILITY_VIOLATION': 2,
   'VAL.TEMPORAL.ORIGIN_AFTER_DESTINATION': 1},
  'checked_fields': ['day_type',
   'destination_h3_index',
   'destination_latitude',
   'destination_longitude',
   'destination_municipality',
   'destination_stop_code',
   'destination_ti


issues:


,level,code,message,field,row_count
0,error,VAL.CORE.OD_SPATIAL_BOTH_MISSING,La regla de OD parcial falló: la fila no tiene ni origen espacial completo ni destino espacial completo en coordenadas.,None,244
1,error,VAL.CORE.NULLABILITY_VIOLATION,"El campo 'origin_h3_index' no admite nulos según nullable efectivo, pero se detectaron filas con valores faltantes.",origin_h3_index,945
2,error,VAL.CORE.NULLABILITY_VIOLATION,"El campo 'destination_h3_index' no admite nulos según nullable efectivo, pero se detectaron filas con valores faltan...",destination_h3_index,1081
3,error,VAL.TEMPORAL.ORIGIN_AFTER_DESTINATION,Se detectaron filas donde origin_time_utc es posterior a destination_time_utc.,None,1


### Resumen corto de issues por código

In [21]:
issues_df = issues_to_frame(report_adatrap_validate.issues)

if not issues_df.empty:
    display(
        issues_df.groupby(["level", "code"])
        .size()
        .reset_index(name="count")
        .sort_values(["level", "count"], ascending=[True, False])
    )
else:
    print("No se emitieron issues.")

,level,code,count
0,error,VAL.CORE.NULLABILITY_VIOLATION,2
1,error,VAL.CORE.OD_SPATIAL_BOTH_MISSING,1
2,error,VAL.TEMPORAL.ORIGIN_AFTER_DESTINATION,1


### Interpretación breve:

- Este caso muestra validate_trips como herramienta diagnóstica sobre un dataset real importado.
- Que report.ok sea False puede ser esperable en esta etapa.
- Lo importante aquí es ver cómo leer issues, summary, eventos e is_validated.
- Esto conversa directamente con el pipeline MVP: import -> validate -> fix/clean/filter -> build_flows.

Nota: en esta etapa del proyecto, datasets reales pueden no quedar validados 
    directamente tras importación. Esto es esperable y precisamente motiva 
    operaciones posteriores de limpieza y filtrado.